In [1]:
"""Dansk eksamens-snydeark til Python-hjaelpere.

Aabn denne fil til eksamen og kopier/rediger kald direkte i bunden eller i en
Python-konsol. Filen printer ikke noget af sig selv.
"""

from pathlib import Path
import sys

import numpy as np
from sympy import *

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

SUPPORT = ROOT / "exam_support"

for path in (ROOT, SUPPORT):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from scripts.circuits.instrumentation import instrumentation_gains
from scripts.circuits.node_equations import solve_node_circuit
from scripts.expand import expand_expr, same_after_expand
from scripts.filters.butterworth import (
    butterworth_highpass_order,
    butterworth_lowpass_order,
    butterworth_poles_q,
    frequency_scale_rc,
)
from scripts.fourier.properties import exponential_transform_variant
from scripts.fourier.series import complex_fourier_coefficients
from scripts.frequency.bode import bode_values, classify_filter_limits, transfer_from_poles_zeros
from scripts.lti.responses import (
    impulse_response_from_transfer,
    inverse_laplace_rational_expr,
    ramp_response_from_transfer,
    related_unit_responses,
    step_response_from_transfer,
    transfer_from_ode,
    zero_input_laplace_second_order,
)
from scripts.lti.second_order import (
    classify_second_order,
    estimate_from_overshoot_peak_time,
    step_features_from_zeta_wn,
)
from scripts.sampling.adc import adc_lsb, alias_frequency, required_sampling_rate_for_lsb
from scripts.transforms.convolution import convolve_causal
from scripts.transforms.laplace import inverse_laplace_rational

s, t, omega, n = symbols("s t omega n", real=True)


In [2]:
# ---------------------------------------------------------------------------
# LAPLACE, TRANSFERFUNKTIONER, RESPONSER
# ---------------------------------------------------------------------------
# inverse_laplace_rational(num, den)
# Input: koefficientlister i faldende potenser af s for num(s)/den(s).
# Brug når: hurtig invers Laplace for rationel funktion.
# Eksempel: 1/(s+1)
# inverse_laplace_rational([1], [1, 1])
#
# inverse_laplace_rational_expr(expr)
# Input: SymPy-udtryk i s.
# Brug når: du allerede har H(s) som formel.
# Eksempel:
# inverse_laplace_rational_expr((s + 2) / (s**2 + 3*s + 2))
#
# inverse_laplace_transform(F, s, t)
# Input: vilkaarlligt SymPy-udtryk F(s), variabler s og t.
# Brug når: direkte SymPy-kald, isaer til opgaver med givne F(s).
# Eksempel:
# inverse_laplace_transform(2*s**2 / (s**2 + 4*s + 4), s, t)
#
# impulse_response_from_transfer(num, den)
# Input: H(s)=num(s)/den(s), koefficienter i faldende potenser.
# Brug når: h(t)=L^-1{H(s)}.
# Eksempel:
# impulse_response_from_transfer([32, 0], [1, 8, 16])
#
# step_response_from_transfer(num, den)
# Input: H(s)=num(s)/den(s).
# Brug når: y_step(t)=L^-1{H(s)/s}.
# Eksempel:
# step_response_from_transfer([1], [1, 2, 1])
#
# ramp_response_from_transfer(num, den)
# Input: H(s)=num(s)/den(s).
# Brug når: y_ramp(t)=L^-1{H(s)/s**2}.
# Eksempel:
# ramp_response_from_transfer([1], [1, 1])
#
# related_unit_responses(response, known)
# Input: response som SymPy-udtryk i t; known = "impulse", "step" eller "ramp".
# Brug når: find impuls/trin/rampe fra en kendt kausal nul-start respons.
# Eksempel:
# related_unit_responses((1 - exp(-t))*Heaviside(t), "step")
#
# zero_input_laplace_second_order(a1, a0, y0, yd0)
# Input: y'' + a1*y' + a0*y = 0, y(0-)=y0, y'(0-)=yd0.
# Brug når: egenrespons/nul-input i Laplace.
# Eksempel:
# zero_input_laplace_second_order(2, 17, 2, 4)
#
# transfer_from_ode(num, den)
# Input: koefficienter for numerator/denominator i H(s).
# Brug når: H(s), nulpunkter og poler fra ODE/transferfunktion.
# Eksempel:
# transfer_from_ode([1], [1, 2, 17])


# ---------------------------------------------------------------------------
# FOLDNING OG ALGEBRA
# ---------------------------------------------------------------------------
# convolve_causal(x1, x2)
# Input: x1(t), x2(t) uden sidste Heaviside(t); funktionen antager kausalitet.
# Brug når: foldning af kausale signaler.
# Eksempel:
# convolve_causal(2*exp(-2*t), t*exp(-3*t))
#
# expand_expr(expr)
# Input: SymPy-udtryk.
# Brug når: normaliser/simplificer/udvid et udtryk, saa MCQ-former kan matches.
# Eksempel:
# expand_expr(((2*t - 4)*exp(0.5*t) + 4)*exp(-0.5*t)*Heaviside(t))
#
# same_after_expand(a, b)
# Input: to SymPy-udtryk.
# Brug når: tjek om to svarformer er algebraisk ens efter simplificering.
# Eksempel:
# same_after_expand(exp(t)*exp(-t), 1)


# ---------------------------------------------------------------------------
# ANDENORDENSSYSTEMER
# ---------------------------------------------------------------------------
# classify_second_order(a1, a0)
# Input: naevner s**2 + a1*s + a0.
# Returnerer: wn, zeta, Q, poles, stable, damping_class.
# Eksempel:
# classify_second_order(2, 17)
#
# step_features_from_zeta_wn(zeta, wn)
# Input: 0 < zeta < 1 og wn > 0.
# Returnerer: wd, percent_overshoot, peak_time, settling_time_2pct.
# Eksempel:
# step_features_from_zeta_wn(0.5, 4.0)
#
# estimate_from_overshoot_peak_time(percent_overshoot, peak_time)
# Input: overshoot i procent og foerste peak-tid.
# Brug når: trinplot giver PO og tp.
# Eksempel:
# estimate_from_overshoot_peak_time(45.59, 0.78543)


# ---------------------------------------------------------------------------
# FOURIER
# ---------------------------------------------------------------------------
# complex_fourier_coefficients(expr, var, period, n_values, t_start=0)
# Input: x(t), variabel, periode T, liste af n-vaerdier, start for perioden.
# Brug når: komplekse Fourier-koefficienter D_n.
# Eksempel:
# complex_fourier_coefficients(1, t, 2*pi, [-1, 0, 1])
#
# exponential_transform_variant(a, time_scale=1, modulation=0)
# Input: a for exp(-a*t)u(t), positiv time_scale, modulation i rad/s.
# Brug når: Fourier-transform-egenskaber for skalering/modulation.
# Eksempel:
# exponential_transform_variant(3, modulation=2)


# ---------------------------------------------------------------------------
# BODE, POLER, NULPUNKTER, FILTRE
# ---------------------------------------------------------------------------
# bode_values(num, den, omega)
# Input: koefficienter for H(s), omega-liste i rad/s.
# Returnerer: H, magnitude_db, phase_deg.
# Eksempel:
# bode_values([1], [1, 1], [0, 1, 10])
#
# classify_filter_limits(num, den)
# Input: koefficienter for rationelt filter.
# Brug når: lavpas/hoejpas ud fra DC- og hoejfrekvensgraenser.
# Eksempel:
# classify_filter_limits([1], [1, 1])
#
# transfer_from_poles_zeros(zeros, poles, gain=1.0)
# Input: lister af nulpunkter, poler og gain.
# Returnerer: num- og den-koefficienter.
# Eksempel:
# transfer_from_poles_zeros([0], [-1, -2], gain=2)
#
# butterworth_lowpass_order(fp, fs, ap_db, as_db)
# Input: passband-frekvens, stopband-frekvens, passband-dB, stopband-dB.
# Krav: 0 < fp < fs.
# Eksempel:
# butterworth_lowpass_order(500, 3000, 3, 72)
#
# butterworth_highpass_order(fp, fs, ap_db, as_db)
# Input: passband-frekvens, stopband-frekvens, passband-dB, stopband-dB.
# Krav: 0 < fs < fp.
# Eksempel:
# butterworth_highpass_order(100, 10, 3, 26)
#
# butterworth_poles_q(order, cutoff=1.0)
# Input: filterorden og cutoff i rad/s.
# Returnerer: stabile Butterworth-poler og Q for 2.-ordenssektioner.
# Eksempel:
# butterworth_poles_q(4, cutoff=1000)
#
# frequency_scale_rc(R, C, omega_c, mode="keep_R")
# Input: normaliseret R, C, target omega_c i rad/s, mode "keep_R" eller "keep_C".
# Brug når: skaler RC fra 1 rad/s til oensket knaekfrekvens.
# Eksempel:
# frequency_scale_rc(1000, 1e-6, 100, mode="keep_R")


# ---------------------------------------------------------------------------
# SAMPLING, ALIASING, ADC
# ---------------------------------------------------------------------------
# adc_lsb(bits, v_min=0.0, v_max=5.0)
# Input: bitantal og spaendingsinterval.
# Returnerer: ideel LSB-stoerrelse.
# Eksempel:
# adc_lsb(16, 0, 5)
#
# alias_frequency(f_signal, f_sample)
# Input: positiv signalfrekvens og samplingsfrekvens i samme enhed.
# Returnerer: foldet frekvens i [0, Fs/2].
# Eksempel:
# alias_frequency(4, 6)
#
# required_sampling_rate_for_lsb(bits, v_range, filter_order, f_3db, worst_amplitude=None)
# Input: ADC bits, fuldt range, filterorden, f_3db, evt. worst-case amplitude.
# Brug når: anti-alias krav, hvor restamplitude skal under 1 LSB.
# Eksempel:
# required_sampling_rate_for_lsb(16, 5, 4, 100)


# ---------------------------------------------------------------------------
# INSTRUMENTERINGSFORSTAERKER
# ---------------------------------------------------------------------------
# instrumentation_gains(R2, RG, alpha=1.0)
# Input: modstande R2, RG og single-mismatch faktor alpha.
# Returnerer: Gd, Gc, CMRR_dB.
# Eksempel:
# instrumentation_gains(499.5, 1, alpha=0.9)
#
# solve_node_circuit(equations, nodes, input_signal=None, output_node=None, s_symbol=s)
# Input: KCL-ligninger i Laplace-domænet, ukendte node-symboler, evt. input/output.
# Returnerer: node_voltages, H, numerator, denominator, coeffs, laplace_equation.
# Brug naar: du har skrevet knudepunktsligninger og vil isolere H(s).
# Eksempel fra notebooken:
# R1, R2, C1, C2 = symbols("R1 R2 C1 C2", positive=True, real=True)
# VA, VB, V1 = symbols("VA VB V1")
# eqA = Eq((VA - V1)*s*C1 + VA/R1 + (VA - VB)/R2, 0)
# eqB = Eq((VB - VA)/R2 + VB*s*C2, 0)
# out = solve_node_circuit([eqA, eqB], [VA, VB], input_signal=V1, output_node=VB, s_symbol=s)
# out["H"]
# out["denominator_coeffs"]


In [3]:
R1, R2, C1, C2 = symbols("R1 R2 C1 C2", positive=True, real=True)
VA, VB, V1 = symbols("VA VB V1")

eqA = Eq((VA - V1) * s*C1 + VA/R1 + (VA-VB)/R2,0)
eqB = Eq((VB - VA)/R2 + VB*s*C2, 0)

solve_node_circuit([eqA, eqB], [VA, VB], input_signal=V1, output_node=VB, s_symbol=s)

{'node_voltages': {VA: (C1*C2*R1*R2*V1*s**2 + C1*R1*V1*s)/(C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1),
  VB: C1*R1*V1*s/(C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1)},
 'raw_solution': {VA: (C1*C2*R1*R2*V1*s**2 + C1*R1*V1*s)/(C1*C2*R1*R2*s**2 + C1*R1*s + C2*R1*s + C2*R2*s + 1),
  VB: C1*R1*V1*s/(C1*C2*R1*R2*s**2 + C1*R1*s + C2*R1*s + C2*R2*s + 1)},
 'H': C1*R1*s/(C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1),
 'numerator': C1*R1*s,
 'denominator': C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1,
 'numerator_coeffs': [C1*R1, 0],
 'denominator_coeffs': [C1*C2*R1*R2, C1*R1 + C2*R1 + C2*R2, 1],
 'laplace_equation': Eq(Y*(C1*C2*R1*R2*s**2 + s*(C1*R1 + C2*R1 + C2*R2) + 1), C1*R1*X*s)}